# DMM-WcycleGAN Full Pipeline Entry

This notebook provides a single entry point for the local workflow under `GAN/DMM-WcycleGAN`:

1. Download the public Neural Latents Benchmark `MC_Maze_Small` dataset from DANDI.
2. Convert the raw NWB files into official NLB `h5` tensors and GAN-ready `npz` exports.
3. Inspect the exported arrays.
4. Run the paper-aligned `DMM-WcycleGAN` model smoke path.

The last section includes a small adapter from NLB sequence tensors to the paper model's `80 x 8` feature map shape. That adapter is only for local end-to-end verification and is not the original paper's closed-source feature extraction.

In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch


def locate_work_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "dataset_loader.py").exists() and (cwd / "DMM-WcycleGAN.py").exists():
        return cwd

    candidate = cwd / "GAN" / "DMM-WcycleGAN"
    if (candidate / "dataset_loader.py").exists():
        return candidate

    raise FileNotFoundError("Could not locate GAN/DMM-WcycleGAN from the current working directory.")


WORK_DIR = locate_work_dir()
REPO_ROOT = WORK_DIR.parent.parent
VENV_PYTHON = REPO_ROOT / ".venv" / "bin" / "python"
DANDI_BIN = REPO_ROOT / ".venv" / "bin" / "dandi"

DATASET_DIR = WORK_DIR / "datasets" / "mc_maze_small"
RAW_DANDI_DIR = DATASET_DIR / "000140"
PIPELINE_DIR = WORK_DIR / "processed" / "mc_maze_small"
OFFICIAL_DIR = PIPELINE_DIR / "official_h5"
GAN_READY_DIR = PIPELINE_DIR / "gan_ready"

print(f"WORK_DIR: {WORK_DIR}")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"PIPELINE_DIR: {PIPELINE_DIR}")
print(f"Python exists: {VENV_PYTHON.exists()}")
print(f"DANDI exists: {DANDI_BIN.exists()}")

WORK_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN
REPO_ROOT: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv
DATASET_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/datasets/mc_maze_small
PIPELINE_DIR: /Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small
Python exists: True
DANDI exists: True


In [2]:
def local_cli_env() -> dict[str, str]:
    env = os.environ.copy()
    env["HOME"] = str(REPO_ROOT)
    env["XDG_CACHE_HOME"] = str(REPO_ROOT / ".cache")
    env["XDG_STATE_HOME"] = str(REPO_ROOT / ".state")
    env["XDG_CONFIG_HOME"] = str(REPO_ROOT / ".config")
    env["PYTHONUNBUFFERED"] = "1"
    return env


DATASET_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DANDI_DIR.exists():
    subprocess.run(
        [
            str(DANDI_BIN),
            "download",
            "-o",
            str(DATASET_DIR),
            "DANDI:000140/draft",
        ],
        cwd=REPO_ROOT,
        env=local_cli_env(),
        check=True,
    )
else:
    print("Dataset already present, skipping download.")

sorted(path.relative_to(DATASET_DIR) for path in DATASET_DIR.rglob("*.nwb"))

Dataset already present, skipping download.


[PosixPath('000140/sub-Jenkins/sub-Jenkins_ses-small_desc-test_ecephys.nwb'),
 PosixPath('000140/sub-Jenkins/sub-Jenkins_ses-small_desc-train_behavior+ecephys.nwb')]

In [3]:
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        str(VENV_PYTHON),
        str(WORK_DIR / "dataset_loader.py"),
        "--verbose",
        "all",
        "--dataset-name",
        "mc_maze_small",
        "--nwb-path",
        str(RAW_DANDI_DIR),
        "--output-dir",
        str(PIPELINE_DIR),
        "--flatten-time",
    ],
    cwd=REPO_ROOT,
    check=True,
)

summary_path = GAN_READY_DIR / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary

INFO: Multiple NWB files detected under `/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/datasets/mc_maze_small/000140`; defaulting to training file `sub-Jenkins_ses-small_desc-train_behavior+ecephys.nwb`.
INFO: Using NLB dataset `mc_maze_small` from `/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/datasets/mc_maze_small/000140/sub-Jenkins/sub-Jenkins_ses-small_desc-train_behavior+ecephys.nwb`
DEBUG: Registering codec 'zlib'
DEBUG: Registering codec 'gzip'
DEBUG: Registering codec 'bz2'
DEBUG: Registering codec 'lzma'
DEBUG: Registering codec 'blosc'
DEBUG: Registering codec 'zstd'
DEBUG: Registering codec 'lz4'
DEBUG: Registering codec 'astype'
DEBUG: Registering codec 'delta'
DEBUG: Registering codec 'quantize'
DEBUG: Registering codec 'fixedscaleoffset'
DEBUG: Registering codec 'packbits'
DEBUG: Registering codec 'categorize'
DEBUG: Registering codec 'pickle'
DEBUG: Registering codec 'base64'
DEBUG: 

{'input_h5': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/official_h5/mc_maze_small_full.h5',
 'flatten_time': True,
 'splits': {'train': {'file': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/gan_ready/train.npz',
   'arrays': {'encoder_input': [75, 140, 107],
    'heldout_target': [75, 140, 35],
    'full_target': [75, 140, 142],
    'forward_target': [75, 40, 142],
    'heldout_mask': [75, 140, 35],
    'forward_mask': [75, 40, 142],
    'extra__behavior': [75, 140, 2],
    'encoder_input_flat': [75, 14980],
    'full_target_flat': [75, 19880]}},
  'eval': {'file': '/Users/fangablt/Applications/EngineeringWorks/testProject/test_newPyEnv/GAN/DMM-WcycleGAN/processed/mc_maze_small/gan_ready/eval.npz',
   'arrays': {'encoder_input': [25, 140, 107],
    'heldout_target': [25, 140, 35],
    'full_target': [25, 140, 142],
    'forward_target': [25, 0, 1

In [4]:
train_npz = np.load(GAN_READY_DIR / "train.npz")
eval_npz = np.load(GAN_READY_DIR / "eval.npz")

train_shapes = {key: train_npz[key].shape for key in train_npz.files}
eval_shapes = {key: eval_npz[key].shape for key in eval_npz.files}

print("Train shapes:")
for key, value in train_shapes.items():
    print(f"  {key}: {value}")

print("\nEval shapes:")
for key, value in eval_shapes.items():
    print(f"  {key}: {value}")

Train shapes:
  encoder_input: (75, 140, 107)
  heldout_target: (75, 140, 35)
  full_target: (75, 140, 142)
  forward_target: (75, 40, 142)
  heldout_mask: (75, 140, 35)
  forward_mask: (75, 40, 142)
  extra__behavior: (75, 140, 2)
  encoder_input_flat: (75, 14980)
  full_target_flat: (75, 19880)

Eval shapes:
  encoder_input: (25, 140, 107)
  heldout_target: (25, 140, 35)
  full_target: (25, 140, 142)
  forward_target: (25, 0, 142)
  heldout_mask: (25, 140, 35)
  forward_mask: (25, 0, 142)
  encoder_input_flat: (25, 14980)
  full_target_flat: (25, 19880)


## Model Smoke Path

The model script uses a hyphenated filename, so we load it with `importlib`. The next cell runs the synthetic smoke test that was already verified from the terminal.

In [5]:
def load_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


dmm_module = load_module("dmm_wcyclegan_module", WORK_DIR / "DMM-WcycleGAN.py")
config = dmm_module.DMMWcycleGANConfig()
dmm_module.run_smoke_test(config)

device: cpu
generator_params: 149826
critic_params: 185602
classifier_params: 101571
generator_loss: 58.27164840698242
critic_loss: 5.901526927947998
classifier_loss: 1.1653109788894653
generator_stats: {'total': 58.27164840698242, 'adv': 0.15295645594596863, 'cycle': 5.105618476867676, 'identity': 1.4125012159347534}
critic_stats: {'total': 5.901526927947998, 'critic_target': -0.03314981609582901, 'critic_source': 0.0017768628895282745, 'gp_target': 0.988595724105835, 'gp_source': 0.9890376329421997}
classifier_stats: {'total': 1.1653109788894653, 'ce': 1.165001392364502, 'reg': 3.0963246822357178}
augmented_feature_shape: (16, 1, 80, 8)
augmented_label_shape: (16,)


## NLB-to-Model Bridge Demo

The exported NLB tensors are spike sequences with shape `(trials, time, units)`, while the paper model currently expects `(batch, 80, 8)` or flattened `640`-dimensional features. The following adapter is only a local bridge for forward-pass verification:

- keep the first `8` time bins
- keep the first `80` channels
- pad with zeros if needed
- transpose to `(batch, channels, bins)`

This is intentionally marked as a demo adapter rather than a claim of paper-faithful feature extraction.

In [6]:
def bridge_nlb_to_paper_features(spike_array: np.ndarray, channels: int = 80, bins: int = 8) -> np.ndarray:
    clipped = spike_array[:, :bins, :channels]
    output = np.zeros((clipped.shape[0], channels, bins), dtype=np.float32)
    output[:, : clipped.shape[2], : clipped.shape[1]] = np.transpose(clipped, (0, 2, 1))
    return output


paper_like_features = bridge_nlb_to_paper_features(train_npz["encoder_input"][:8])
paper_like_labels = torch.zeros(paper_like_features.shape[0], dtype=torch.long)

model = dmm_module.DMMWcycleGAN(config)
with torch.no_grad():
    logits = model.classify(torch.from_numpy(paper_like_features))

print("paper_like_features:", paper_like_features.shape)
print("logits:", tuple(logits.shape))
print("predictions:", logits.argmax(dim=1).tolist())
paper_like_labels

paper_like_features: (8, 80, 8)
logits: (8, 3)
predictions: [0, 1, 2, 2, 0, 0, 0, 0]


tensor([0, 0, 0, 0, 0, 0, 0, 0])